# Предобработка текста

In [1]:
!pip install pdfplumber
!pip install langchain_huggingface
!pip install langchain_community
!pip install sentence-transformers
!apt-get -y update >/dev/null
!apt-get -y install fonts-dejavu-core >/dev/null

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 10.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/

In [2]:
import pdfplumber

pdf_path = "/content/drive/MyDrive/Colab Notebooks/NLP_llm/langchain/instr.pdf"

# pages_to_parse = [18, 23]   # страницы таблиц
pages_to_parse =  range(7, 24)

all_tables = []
all_text = []

def words_to_lines(words, y_tol=3.0):
    if not words:
        return []
    words = sorted(words, key=lambda w: (w["top"], w["x0"]))
    lines, cur, cur_top = [], [], words[0]["top"]

    for w in words:
        if abs(w["top"] - cur_top) <= y_tol:
            cur.append(w)
        else:
            cur = sorted(cur, key=lambda x: x["x0"])
            lines.append(" ".join(x["text"] for x in cur))
            cur = [w]
            cur_top = w["top"]

    cur = sorted(cur, key=lambda x: x["x0"])
    lines.append(" ".join(x["text"] for x in cur))
    return lines

def clean(text: str) -> str:
    text = text.replace("\u00ad", "")  # soft hyphen
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def inside(w, bb, pad=1.5):
    x0, top, x1, bottom = bb
    cx = (w["x0"] + w["x1"]) / 2
    cy = (w["top"] + w["bottom"]) / 2
    return (x0-pad) <= cx <= (x1+pad) and (top-pad) <= cy <= (bottom+pad)

with pdfplumber.open(pdf_path) as pdf:

    for page_idx in pages_to_parse:
        page = pdf.pages[page_idx]
        # im = page.to_image(resolution=170)   # если нужно визуализировать

        # ---- 1. Ищем таблицы ----
        tables = page.find_tables()

        for table in tables:
            # im.draw_rects(table.cells, stroke="red", fill=None)  # подсветка таблиц
            all_tables.append(table.extract())  # сохраняем извлечённую таблицу

        # ---- 2. Получаем bbox всех таблиц ----
        bboxes = [t.bbox for t in tables]   # bbox каждой таблицы

        # for t in tables:
        #     im.draw_rects(t.cells, stroke="red", fill=None)

        # ---- 3. Извлекаем весь текст страницы ----
        words = page.extract_words(use_text_flow=True)

        # ---- 4. Фильтруем слова, которые попали внутрь таблиц ----
        words_no_tables = [
            w for w in words
            if not any(inside(w, bb) for bb in bboxes)
        ]

        # ---- 5. (Опционально) визуализация слов ----
        # rects = [(w["x0"], w["top"], w["x1"], w["bottom"]) for w in words_no_tables]
        # im.draw_rects(rects, stroke="blue", fill=None)
        # display(im)

        # ---- 6. Собираем текст страницы ----
        words_sorted = sorted(words_no_tables, key=lambda w: (w["top"], w["x0"]))

        lines = []
        current_line = []
        current_top = None
        y_tol = 3.0  # чувствительность к разбросу по высоте строки

        for w in words_sorted:
            if current_top is None or abs(w["top"] - current_top) <= y_tol:
                current_line.append(w)
                current_top = w["top"] if current_top is None else current_top
            else:
                current_line = sorted(current_line, key=lambda x: x["x0"])
                lines.append(" ".join(x["text"] for x in current_line))
                current_line = [w]
                current_top = w["top"]

        if current_line:
            current_line = sorted(current_line, key=lambda x: x["x0"])
            lines.append(" ".join(x["text"] for x in current_line))

        page_text = "\n".join(lines).strip()

        # сохраняем текст страницы (если нужен)
        all_text.append(page_text)

len(all_tables)



2

In [3]:
import pandas as pd
import numpy as np

table = all_tables[0]
df = pd.DataFrame(table[1:], columns=table[0])

# 0) привести None / "None" к NaN
df = df.replace("None", np.nan)

# 1) ffill вниз по "Характеристика"
df["Характеристика"] = df["Характеристика"].ffill()

# 2) склейка "подпункта" (2-я колонка) в "Характеристика"
#    делаем это ТОЛЬКО если реально есть лишняя колонка между "Характеристика" и "ПКФ-80"
#    (как у тебя на скрине)
subcol_index = 1
if df.shape[1] > subcol_index:
    sub = df.iloc[:, subcol_index]              # гарантированно Series
    mask = sub.notna()

    df.loc[mask, "Характеристика"] = (
        df.loc[mask, "Характеристика"].astype(str).str.strip()
        + " — "
        + sub[mask].astype(str).str.strip()
    )

    # 3) удалить колонку подпункта по позиции (без бага с одинаковыми именами)
    col_to_drop = df.columns[subcol_index]
    df = df.drop(columns=[col_to_drop])

# 4) убрать полностью пустые колонки (и где все NaN)
df = df.dropna(axis=1, how="all")
df.iloc[-1, -1] = 572
df = df.ffill(axis=1)

df.to_json()


'{"\\u0425\\u0430\\u0440\\u0430\\u043a\\u0442\\u0435\\u0440\\u0438\\u0441\\u0442\\u0438\\u043a\\u0430":{"0":"\\u0412\\u043c\\u0435\\u0441\\u0442\\u0438\\u043c\\u043e\\u0441\\u0442\\u044c \\u0444\\u0438\\u043b\\u044c\\u0442\\u0440\\u0430, \\u043c3","1":"\\u041f\\u043b\\u043e\\u0449\\u0430\\u0434\\u044c \\u043f\\u043e\\u0432\\u0435\\u0440\\u0445\\u043d\\u043e\\u0441\\u0442\\u0438 \\u0444\\u0438\\u043b\\u044c\\u0442\\u0440\\u043e\\u0432\\u0430\\u043d\\u0438\\u044f, \\u043c2","2":"\\u041a\\u043e\\u043b\\u0438\\u0447\\u0435\\u0441\\u0442\\u0432\\u043e \\u0444\\u0438\\u043b\\u044c\\u0442\\u0440\\u043e\\u0432\\u0430\\u043b\\u044c\\u043d\\u044b\\u0445 \\u043f\\u0430\\u0442\\u0440\\u043e\\u043d\\u043e\\u0432, \\u0448\\u0442.","3":"\\u0413\\u0430\\u0431\\u0430\\u0440\\u0438\\u0442\\u043d\\u044b\\u0435 \\u0440\\u0430\\u0437\\u043c\\u0435\\u0440\\u044b,\\n\\u043c\\u043c: \\u2014 \\u0434\\u0438\\u0430\\u043c\\u0435\\u0442\\u0440","4":"\\u0413\\u0430\\u0431\\u0430\\u0440\\u0438\\u0442\\u043d\\u044b\

In [4]:
table2 = all_tables[1]
df2 = pd.DataFrame(table2[1:], columns=table2[0])

model_name = df2.columns[1]

lines = [
    f"{model_name} {row['Характеристика']} : {row[model_name]}"
    for _, row in df2.iterrows()
]

result_text = "\n".join(lines)
result_text



'ДТ68-2,5 Поверхность фильтрования, м2 : 68\nДТ68-2,5 Вакуум в зоне фильтрования и сушки, кгс/см2 (Па) : от 0,25 до 0,5 (от25 до50)\nДТ68-2,5 Давление воздуха в зоне отдувки, МПа : от 0,15 до 0,24\nДТ68-2,5 Диаметр дисков, мм : 2500\nДТ68-2,5 Количество дисков, шт. : 8\nДТ68-2,5 Количество секторов в диске, шт. : 12\nДТ68-2,5 Скорость вращения дисков, об/мин : от 0,22 до 0,97\nДТ68-2,5 Материал частей фильтра, соприкасающихся с\nпродуктом фильтрования : сплав\nВТI-0'

In [5]:
#

In [6]:
doc_text = "\n\n".join(all_text)
print("Pages:", len(all_text))
print("Doc length:", len(doc_text))
print(doc_text[:1000])


Pages: 17
Doc length: 33693
ТИ 305.2-01-2022
Введение
Настоящая технологическая инструкция регламентирует
последовательность технологических операций приготовления никелевого
католита, правила ведения и управления процессом, содержит характеристику
сырья, материалов, оборудования и товарной продукции, методы контроля и
метрологическое обеспечение, а также требования охраны труда и
промышленной безопасности.
Инструкция разработана в соответствии с проектом ФМ.04450-П-ИОС7.1-С
«ЦЭН. ОЭН-2. Электроэкстракция никеля из растворов хлорного растворения
ПНТП на объем производства 145 тыс.т/год электролитного никеля»,
выполненным в 2015 году ООО «Институт Гипроникель» на основании
Технологического регламента, выполненного по договору № 001-134н/0438-49-
14 от 01.04.2014 г. «Актуализация технологического регламента на
корректировку проекта переработки никелевого порошка трубчатых печей по
технологии хлорного выщелачивания с увеличением мощности ЦЭН-2 ОАО
«Кольская ГМК» производства 145 тыс. т/го

In [7]:
import re

doc_text = "\n\n".join(p.strip() for p in all_text if p and p.strip())

# удалить номера листов
doc_text = re.sub(r"Лист\s+\d+\s+Листов\s+\d+", "", doc_text)

# убрать множественные переносы
doc_text = re.sub(r"\n{3,}", "\n\n", doc_text)
doc_text = re.sub(r"Рисунок\s+\d+.*", "", doc_text)
doc_text = re.sub(r"ТИ\s+305\.2-01-2022", "", doc_text)
doc_text = re.sub(r"Лист\s+\d+\s+Листов\s+\d+", "", doc_text)
doc_text = doc_text.strip()
doc_text

'Введение\nНастоящая технологическая инструкция регламентирует\nпоследовательность технологических операций приготовления никелевого\nкатолита, правила ведения и управления процессом, содержит характеристику\nсырья, материалов, оборудования и товарной продукции, методы контроля и\nметрологическое обеспечение, а также требования охраны труда и\nпромышленной безопасности.\nИнструкция разработана в соответствии с проектом ФМ.04450-П-ИОС7.1-С\n«ЦЭН. ОЭН-2. Электроэкстракция никеля из растворов хлорного растворения\nПНТП на объем производства 145 тыс.т/год электролитного никеля»,\nвыполненным в 2015 году ООО «Институт Гипроникель» на основании\nТехнологического регламента, выполненного по договору № 001-134н/0438-49-\n14 от 01.04.2014 г. «Актуализация технологического регламента на\nкорректировку проекта переработки никелевого порошка трубчатых печей по\nтехнологии хлорного выщелачивания с увеличением мощности ЦЭН-2 ОАО\n«Кольская ГМК» производства 145 тыс. т/год катодного никеля, включая\n

# Retriveing


In [8]:
questions = [
    "Что регламентирует технологическая инструкция ТИ 305.2-01-2022?",
    "Какова основная задача ГМО-2?",
    "Что такое католит согласно тексту?",
    "Что такое анолит согласно тексту?",
    "Из каких операций состоит производство католита?",
    "Какой химический состав исходного раствора ОРиД указан в инструкции?",
    "Какая концентрация и плотность серной кислоты используются и куда она сливается?",
    "Как описан хлор: его свойства, подача и хранение?",
    "Какова роль борной кислоты и как она вводится?",
    "Какие основные параметры и ограничения режима у пачука?"
]

In [9]:
answers = [
    "Последовательность технологических операций приготовления никелевого католита, правила ведения и управления процессом, характеристику сырья, материалов, оборудования, продукции, методы контроля и требования охраны труда и промышленной безопасности.",
    "Приготовление католита для производства электролитного никеля.",
    "Электролит, полученный из анолита после очистки от примесей железа, цинка, меди, кобальта и свинца.",
    "Раствор, полученный при электрохимическом растворении НПТП в реакторах растворения ОРиД.",
    "Фильтрация растворов, очистка от железа, цинка, меди, кобальта и свинца, приготовление карбоната никеля, очистка сточных вод, репульпация железистого кека, производство кобальтового концентрата.",
    "Ni2+ ≥ 220; Fe2+ ≤ 8.0; Cu2+ ≤ 2.5; Co2+ ≤ 6.0; Na+ + K+ ≤ 8.5; Cl− 240–260; SO4 55–75; Zn2+ ≤ 0.0025; Pb2+ ≤ 0.032; H2SO4 0.2–1.0.",
    "Концентрация 92.5–94.0%; плотность 1830–1840 кг/м3; сливается в приёмные баки №901–904 ГМО-2.",
    "Газ жёлто-зелёного цвета с резким запахом; тяжелее воздуха; поступает в сжиженном виде в ж/д цистернах, хранится в танках, подаётся в газообразном виде; один трубопровод рабочий, второй резервный.",
    "Буферная добавка для стабилизации концентрации ионов водорода; вводится через репульпатор №201 примерно 1000 кг в сутки.",
    "Объём 170 м3; коэффициент заполнения 0.85; температура не более 85 °C; используется как реакционная ёмкость и накопитель."
]

In [10]:
import re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# 0) doc_text
doc_text = "\n\n".join(p.strip() for p in all_text if p and p.strip())
doc_text = re.sub(r"Лист\s+\d+\s+Листов\s+\d+", " ", doc_text)
doc_text = re.sub(r"ТИ\s*305\.2-01-2022", " ", doc_text)
doc_text = re.sub(r"\n{3,}", "\n\n", doc_text).strip()

# 1) chunking
HEADER_RE = r"\n(?=(?:\d+(?:\.\d+)*|\d+)\s+[А-ЯA-ZЁ])"
def chunk_by_headers(text, min_chars=250):
    parts = re.split(HEADER_RE, "\n" + text)
    return [p.strip() for p in parts if len(p.strip()) >= min_chars]

doc_chunks = chunk_by_headers(doc_text, min_chars=250)

In [11]:
gold_manual = {
  1: 0,   # Введение
  2: 0,
  3: 0,
  4: 0,
  5: 0,
  6: 1,   # 1.1 Сырьё
  7: 2,   # 1.2.1 Серная кислота
  8: 4,   # 1.2.3 Хлор
  9: 5,   # 1.2.4 Борная кислота
  10: 12  # 3.1 Пачук
}


# 3) semantic gold + eval
def build_semantic_gold(answers_for_gold, doc_emb, model):
    a_emb = model.encode(answers_for_gold, normalize_embeddings=True, show_progress_bar=False).astype(np.float32)
    gold = {}
    for qid in range(len(answers_for_gold)):
        scores = (doc_emb @ a_emb[qid].reshape(-1, 1)).ravel()
        gold[qid+1] = int(np.argmax(scores))
    return gold

def eval_vs_manual_gold(questions, doc_chunks, doc_emb, model,
                        gold_manual, ks=(1,3,5), qtext=lambda x:x):
    max_k = max(ks)
    hit = {k: 0 for k in ks}
    mrr_sum = 0.0

    for qid, q in enumerate(questions, start=1):
        g = gold_manual[qid]

        q_emb = model.encode([qtext(q)],
                             normalize_embeddings=True,
                             show_progress_bar=False).astype(np.float32)

        scores = (doc_emb @ q_emb.T).ravel()
        idx = np.argsort(-scores)[:max_k]
        top_ids = [int(i) for i in idx]

        for k in ks:
            hit[k] += int(g in top_ids[:k])

        if g in top_ids:
            mrr_sum += 1.0 / (top_ids.index(g) + 1)

    recalls = {f"Recall@{k}": hit[k]/len(questions) for k in ks}
    mrr = mrr_sum/len(questions)
    return recalls, mrr


In [15]:
models_to_try = [
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "intfloat/multilingual-e5-base",
    "intfloat/multilingual-e5-large",
    "BAAI/bge-m3",
    # "Alibaba-NLP/gte-multilingual-base",
]

results = []

for MODEL_NAME in models_to_try:
    print("Running:", MODEL_NAME)
    model = SentenceTransformer(MODEL_NAME)

    def qtext(q):
        return ("query: " + q) if "multilingual-e5" in MODEL_NAME else q

    def ptext(p):
        return ("passage: " + p) if "multilingual-e5" in MODEL_NAME else p

    # doc embeddings
    doc_emb = model.encode(
        [ptext(c) for c in doc_chunks],
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype(np.float32)

    recalls, mrr = eval_vs_manual_gold(
        questions,
        doc_chunks,
        doc_emb,
        model,
        gold_manual,
        ks=(1,3,5),
        qtext=qtext
    )

    results.append({
        "model": MODEL_NAME,
        **recalls,
        "MRR@5": mrr
    })

df_models = pd.DataFrame(results).sort_values("Recall@1", ascending=False)
display(df_models)

In [12]:
questions_hard = [
    "Какие аспекты производственного процесса описывает данный регламент?",
    "Для какой цели функционирует гидрометаллургическое отделение №2?",
    "Как называется очищенный электролит, поступающий в электролиз?",
    "Какой раствор образуется при растворении НПТП в реакторах ОРиД?",
    "Перечисли технологические стадии подготовки электролита перед электролизом.",
    "Какие ограничения по концентрациям компонентов установлены для исходного раствора?",
    "Какие параметры качества и хранения предъявляются к концентрированной H2SO4?",
    "Какие физические свойства и схема подачи хлора предусмотрены технологией?",
    "Как обеспечивается стабилизация кислотности в прикатодной зоне?",
    "Какие конструктивные и режимные параметры характерны для реакционной ёмкости 170 м3?"
]

In [19]:
import numpy as np
import pandas as pd

terms = [
    "HCl",
    "соляная",
    "соляная кислота",
    "кислота соляная",          # перестановка
    "хлороводородная кислота",  # более научное
    "хлороводород",
    "гидрохлорид",              # НЕ то же самое (часто соль)
    "hydrochloric acid",
    "hydrogen chloride",
    "chloride",                 # тоже НЕ то же
]

# 1) эмбеддинги (обязательно нормализуем => dot == cosine)
emb = model.encode(terms, normalize_embeddings=True, show_progress_bar=False).astype(np.float32)

# 2) матрица cosine similarity
S = emb @ emb.T
df = pd.DataFrame(S, index=terms, columns=terms)

# красиво: округлим
display(df.round(3))

# 3) топ похожих для каждого терма (кроме самого себя)
def top_similar(i, top_k=5):
    scores = S[i].copy()
    scores[i] = -1  # исключаем self
    idx = np.argsort(-scores)[:top_k]
    return [(terms[j], float(scores[j])) for j in idx]

for i, t in enumerate(terms):
    print("\n===", t, "===")
    for cand, sc in top_similar(i, top_k=5):
        print(f"{sc: .3f}  {cand}")

,HCl,соляная,соляная кислота,кислота соляная,хлороводородная кислота,хлороводород,гидрохлорид,hydrochloric acid,hydrogen chloride,chloride
HCl,1.000,0.410,0.510,0.506,0.565,0.623,0.658,0.660,0.689,0.597
соляная,0.410,1.000,0.833,0.831,0.482,0.454,0.391,0.471,0.399,0.460
соляная кислота,0.510,0.833,1.000,0.979,0.699,0.582,0.478,0.604,0.522,0.544
кислота соляная,0.506,0.831,0.979,1.000,0.693,0.578,0.470,0.582,0.513,0.539
хлороводородная кислота,0.565,0.482,0.699,0.693,1.000,0.870,0.595,0.675,0.659,0.696
хлороводород,0.623,0.454,0.582,0.578,0.870,1.000,0.689,0.696,0.703,0.788
гидрохлорид,0.658,0.391,0.478,0.470,0.595,0.689,1.000,0.775,0.764,0.665
hydrochloric acid,0.660,0.471,0.604,0.582,0.675,0.696,0.775,1.000,0.778,0.668
hydrogen chloride,0.689,0.399,0.522,0.513,0.659,0.703,0.764,0.778,1.000,0.805
chloride,0.597,0.460,0.544,0.539,0.696,0.788,0.665,0.668,0.805,1.000



=== HCl ===
 0.689  hydrogen chloride
 0.660  hydrochloric acid
 0.658  гидрохлорид
 0.623  хлороводород
 0.597  chloride

=== соляная ===
 0.833  соляная кислота
 0.831  кислота соляная
 0.482  хлороводородная кислота
 0.471  hydrochloric acid
 0.460  chloride

=== соляная кислота ===
 0.979  кислота соляная
 0.833  соляная
 0.699  хлороводородная кислота
 0.604  hydrochloric acid
 0.582  хлороводород

=== кислота соляная ===
 0.979  соляная кислота
 0.831  соляная
 0.693  хлороводородная кислота
 0.582  hydrochloric acid
 0.578  хлороводород

=== хлороводородная кислота ===
 0.870  хлороводород
 0.699  соляная кислота
 0.696  chloride
 0.693  кислота соляная
 0.675  hydrochloric acid

=== хлороводород ===
 0.870  хлороводородная кислота
 0.788  chloride
 0.703  hydrogen chloride
 0.696  hydrochloric acid
 0.689  гидрохлорид

=== гидрохлорид ===
 0.775  hydrochloric acid
 0.764  hydrogen chloride
 0.689  хлороводород
 0.665  chloride
 0.658  HCl

=== hydrochloric acid ===
 0.778  hyd

In [27]:
import numpy as np
from sentence_transformers import SentenceTransformer

models_to_try = [
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "intfloat/multilingual-e5-base",
    "intfloat/multilingual-e5-large",
    "BAAI/bge-m3",
]

pairs = [("HCl", "соляная"), ("HCl", "соляная кислота"), ("HCl", "hydrochloric acid")]

res = []

for model_name in models_to_try:
    model = SentenceTransformer(model_name)

    def wrap(s):
        # одинаковый режим для сравнения: query-query
        if "multilingual-e5" in model_name:
            return "query: " + s
        return s

    for a, b in pairs:
        v1 = model.encode([wrap(a)], normalize_embeddings=True)
        v2 = model.encode([wrap(b)], normalize_embeddings=True)
        score = float((v1 @ v2.T).item())
        res.append({"model": model_name, "a": a, "b": b, "cos": score})

# печать в таблице
import pandas as pd
df = pd.DataFrame(res).sort_values(["a","b","cos"], ascending=[True, True, False])
display(df)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

,model,a,b,cos
8,intfloat/multilingual-e5-large,HCl,hydrochloric acid,0.907102
5,intfloat/multilingual-e5-base,HCl,hydrochloric acid,0.887596
11,BAAI/bge-m3,HCl,hydrochloric acid,0.659727
2,sentence-transformers/paraphrase-multilingual-...,HCl,hydrochloric acid,0.420199
6,intfloat/multilingual-e5-large,HCl,соляная,0.798984
3,intfloat/multilingual-e5-base,HCl,соляная,0.781186
9,BAAI/bge-m3,HCl,соляная,0.409780
0,sentence-transformers/paraphrase-multilingual-...,HCl,соляная,0.334802
7,intfloat/multilingual-e5-large,HCl,соляная кислота,0.840245
4,intfloat/multilingual-e5-base,HCl,соляная кислота,0.803920


In [28]:
res

[{'model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
  'a': 'HCl',
  'b': 'соляная',
  'cos': 0.3348015248775482},
 {'model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
  'a': 'HCl',
  'b': 'соляная кислота',
  'cos': 0.39414533972740173},
 {'model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
  'a': 'HCl',
  'b': 'hydrochloric acid',
  'cos': 0.42019912600517273},
 {'model': 'intfloat/multilingual-e5-base',
  'a': 'HCl',
  'b': 'соляная',
  'cos': 0.7811857461929321},
 {'model': 'intfloat/multilingual-e5-base',
  'a': 'HCl',
  'b': 'соляная кислота',
  'cos': 0.8039197325706482},
 {'model': 'intfloat/multilingual-e5-base',
  'a': 'HCl',
  'b': 'hydrochloric acid',
  'cos': 0.8875957727432251},
 {'model': 'intfloat/multilingual-e5-large',
  'a': 'HCl',
  'b': 'соляная',
  'cos': 0.798983633518219},
 {'model': 'intfloat/multilingual-e5-large',
  'a': 'HCl',
  'b': 'соляная кислота',
  'cos': 0.8402448296546936},
 {'model': 'int

# Сравнение хим формул

In [29]:
import re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

MODELS = {
    "bge-m3": "BAAI/bge-m3",
    "e5-base": "intfloat/multilingual-e5-base",
    "e5-large": "intfloat/multilingual-e5-large",
}

# Набор тестов (пары + "похожие, но не то")
tests = {
    "H2SO4": [
        "серная кислота", "кислота серная", "sulfuric acid",
        "сульфат", "SO4", "сульфат-ионы", "сульфаты"
    ],
    "NaOH": [
        "гидроксид натрия", "едкий натр", "sodium hydroxide",
        "сода кальцинированная", "Na2CO3", "щёлочь"
    ],
    "Na2CO3": [
        "кальцинированная сода", "карбонат натрия", "sodium carbonate",
        "пищевая сода", "NaHCO3", "сода"
    ],
    "Cl2": [
        "хлор", "газообразный хлор", "chlorine",
        "хлорид", "chloride", "Cl-"
    ],
    "Cl-": [
        "хлорид", "ионы хлора", "chloride ion",
        "хлор", "Cl2"
    ],
    "SO4^2-": [
        "сульфат", "сульфат-ион", "sulfate ion",
        "серная кислота", "H2SO4"
    ],
    "Ni2+": [
        "никель", "ионы никеля", "nickel ion",
        "Ni", "никелевый"
    ],
    "Fe2+": [
        "железо(II)", "двухвалентное железо", "iron(II) ion",
        "Fe3+", "железо(III)"
    ],
    "Cu2+": [
        "медь(II)", "ионы меди", "copper(II) ion",
        "Cu", "медный"
    ],
    "Co2+": [
        "кобальт(II)", "ионы кобальта", "cobalt(II) ion",
        "Co", "кобальтовый"
    ],
    "Zn2+": [
        "цинк(II)", "ионы цинка", "zinc(II) ion",
        "Zn", "цинковый"
    ],
    "Pb2+": [
        "свинец(II)", "ионы свинца", "lead(II) ion",
        "Pb", "свинцовый"
    ],
}

def cosine(a, b):
    a = a / np.linalg.norm(a, axis=1, keepdims=True)
    b = b / np.linalg.norm(b, axis=1, keepdims=True)
    return (a @ b.T).item()

def wrap_e5(model_name, s, kind="query"):
    # Для сравнения "терм-терм" можно использовать query-query, главное одинаково.
    if "multilingual-e5" in model_name:
        return f"{kind}: {s}"
    return s

rows = []

for label, model_name in MODELS.items():
    model = SentenceTransformer(model_name)

    for formula, candidates in tests.items():
        # enc formula
        f = wrap_e5(model_name, formula, kind="query")
        vf = model.encode([f], show_progress_bar=False).astype(np.float32)

        for cand in candidates:
            c = wrap_e5(model_name, cand, kind="query")
            vc = model.encode([c], show_progress_bar=False).astype(np.float32)

            # normalize manually to be version-safe
            score = cosine(vf, vc)

            rows.append({
                "model": label,
                "formula": formula,
                "candidate": cand,
                "cos": float(score),
            })

df = pd.DataFrame(rows)
# Топ-5 кандидатов для каждой формулы и модели
top5 = (df.sort_values(["model","formula","cos"], ascending=[True, True, False])
          .groupby(["model","formula"])
          .head(5))
display(top5)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,model,formula,candidate,cos
29,bge-m3,Cl-,Cl2,0.702695
27,bge-m3,Cl-,chloride ion,0.539934
28,bge-m3,Cl-,хлор,0.476015
25,bge-m3,Cl-,хлорид,0.467149
26,bge-m3,Cl-,ионы хлора,0.446726
...,...,...,...,...
188,e5-large,Zn2+,Zn,0.920234
187,e5-large,Zn2+,zinc(II) ion,0.913804
185,e5-large,Zn2+,цинк(II),0.868694
186,e5-large,Zn2+,ионы цинка,0.819157


# Разные формулы

In [30]:
import re

CHEM_CANON_MAP = {
    # кислоты/щёлочи
    "h2so4": "серная кислота",
    "hcl": "соляная кислота",
    "naoh": "гидроксид натрия",
    "na2co3": "кальцинированная сода",

    # газы/ионы
    "cl2": "хлор",
    "cl-": "хлорид",
    "so4": "сульфат",
    "so4^2-": "сульфат",
    "so4 2-": "сульфат",
    "so42-": "сульфат",

    # типовые катионы
    "ni2+": "ионы никеля",
    "fe2+": "ионы железа(II)",
    "cu2+": "ионы меди(II)",
    "co2+": "ионы кобальта(II)",
    "zn2+": "ионы цинка",
    "pb2+": "ионы свинца",
}

def canon_chem_text(s: str) -> str:
    """
    Нормализует хим. обозначения + типовые OCR-разрывы.
    Безопасно применять к query и chunk text (лучше к query).
    """
    t = s.lower()

    # унифицируем минусы и пробелы
    t = t.replace("−", "-").replace("–", "-")
    t = re.sub(r"\s+", " ", t)

    # лечим частые разрывы OCR/верстки:
    t = re.sub(r"h\s*2\s*so\s*4", "h2so4", t)
    t = re.sub(r"na\s*2\s*co\s*3", "na2co3", t)
    t = re.sub(r"n\s*a\s*o\s*h", "naoh", t)
    t = re.sub(r"c\s*l\s*2", "cl2", t)
    t = re.sub(r"c\s*l\s*-\s*", "cl-", t)

    # SO4 часто разбито: "SO 2- 4" / "SO 4"
    t = re.sub(r"so\s*4", "so4", t)
    t = re.sub(r"so\s*4\s*\^?\s*2\s*-\s*", "so4^2-", t)

    # ионы могут быть с пробелами
    t = re.sub(r"ni\s*2\s*\+", "ni2+", t)
    t = re.sub(r"fe\s*2\s*\+", "fe2+", t)
    t = re.sub(r"cu\s*2\s*\+", "cu2+", t)
    t = re.sub(r"co\s*2\s*\+", "co2+", t)
    t = re.sub(r"zn\s*2\s*\+", "zn2+", t)
    t = re.sub(r"pb\s*2\s*\+", "pb2+", t)

    # замены по словарю (границы слова где уместно)
    # сначала длинные ключи, чтобы h2so4 не ломалось на so4
    for k in sorted(CHEM_CANON_MAP.keys(), key=len, reverse=True):
        v = CHEM_CANON_MAP[k]
        # аккуратно: формулы могут быть частью текста
        t = re.sub(rf"(?<![a-z0-9]){re.escape(k)}(?![a-z0-9])", v, t)

    return t

In [35]:

def search_similar(query: str, chunks: list[str], chunk_emb: np.ndarray, model, top_k: int = 5):
    q_emb = model.encode([query], normalize_embeddings=True, show_progress_bar=False).astype(np.float32)  # (1, d)
    scores = (chunk_emb @ q_emb.T).ravel()  # cosine because normalized
    idx = np.argsort(-scores)[:top_k]
    return [(int(i), float(scores[i]), chunks[i]) for i in idx]

def search_similar_norm(query: str, chunks, chunk_emb, model, top_k=5):
    qn = canon_chem_text(query)
    q_emb = model.encode([qn], normalize_embeddings=True, show_progress_bar=False).astype(np.float32)
    scores = (chunk_emb @ q_emb.T).ravel()
    idx = np.argsort(-scores)[:top_k]
    return [(int(i), float(scores[i]), chunks[i]) for i in idx]

In [36]:
def eval_retrieval(search_fn, questions, gold_manual, top_k=5):
    hit1=hit3=hit5=0
    mrr=0.0
    for qid, q in enumerate(questions, start=1):
        g = gold_manual[qid]
        res = search_fn(q, doc_chunks, doc_emb, model, top_k=top_k)
        ids = [cid for cid,_,_ in res]
        hit1 += int(g in ids[:1])
        hit3 += int(g in ids[:3])
        hit5 += int(g in ids[:5])
        if g in ids:
            mrr += 1.0/(ids.index(g)+1)
    n=len(questions)
    return {"Recall@1":hit1/n,"Recall@3":hit3/n,"Recall@5":hit5/n,"MRR":mrr/n}

# пример (на bge-m3)
model = SentenceTransformer("BAAI/bge-m3")
doc_emb = model.encode(doc_chunks, normalize_embeddings=True, show_progress_bar=False).astype(np.float32)

base = eval_retrieval(search_similar, questions_hard, gold_manual, top_k=5)
norm = eval_retrieval(search_similar_norm, questions_hard, gold_manual, top_k=5)

print("BASE:", base)
print("NORM:", norm)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BASE: {'Recall@1': 0.6, 'Recall@3': 0.8, 'Recall@5': 0.9, 'MRR': 0.725}
NORM: {'Recall@1': 0.7, 'Recall@3': 0.8, 'Recall@5': 0.9, 'MRR': 0.775}


# Сложные вопросы

In [ ]:
import re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# -------------------------
# 0) ВОПРОСЫ ДЛЯ ЭКСПЕРИМЕНТА (химические)
# -------------------------
chem_questions = [
    "Какая норма по Cl- в исходном растворе?",
    "Какой диапазон по SO4^2- в исходном растворе?",
    "Какой предел для Zn2+ в исходном растворе?",
    "Какое ограничение по Pb2+ в исходном растворе?",
    "Какой диапазон по H2SO4 в исходном растворе?",
]

# gold_manual: словарь {qid: chunk_id}
# ВАЖНО: у тебя уже есть gold_manual для questions_hard.
# Для chem_questions сделаем gold автоматически по ответу через семантический gold (см. ниже).

# -------------------------
# 1) НОРМАЛИЗАЦИЯ ХИМИИ (запрос)
# -------------------------
CHEM_CANON_MAP = {
    "h2so4": "серная кислота",
    "hcl": "соляная кислота",
    "naoh": "гидроксид натрия",
    "na2co3": "кальцинированная сода",

    "cl2": "хлор",       # молекулярный хлор
    "cl-": "хлорид",     # ион
    "so4^2-": "сульфат",
    "so42-": "сульфат",
    "so4": "сульфат",

    "ni2+": "ионы никеля",
    "fe2+": "ионы железа(II)",
    "cu2+": "ионы меди(II)",
    "co2+": "ионы кобальта(II)",
    "zn2+": "ионы цинка",
    "pb2+": "ионы свинца",
}

def canon_chem_text_safe(text: str) -> str:
    t = text.lower()
    t = t.replace("−", "-").replace("–", "-")

    SAFE_RULES = [
        (r"\bcl\s*-\b", "хлорид"),
        (r"\bh\s*2\s*so\s*4\b", "серная кислота"),
        (r"\bzn\s*2\s*\+\b", "ионы цинка"),
        (r"\bpb\s*2\s*\+\b", "ионы свинца"),
    ]

    for pattern, synonym in SAFE_RULES:
        t = re.sub(pattern, lambda m: m.group(0) + " " + synonym, t)

    return t

def canon_add_synonyms(q: str) -> str:
    t = q.lower()
    t = t.replace("−", "-").replace("–", "-")

    # лечим OCR-разрывы (тут можно REPLACE)
    t = re.sub(r"h\s*2\s*so\s*4", "h2so4", t)
    t = re.sub(r"so\s*4", "so4", t)
    t = re.sub(r"cl\s*-\s*", "cl-", t)
    t = re.sub(r"zn\s*2\s*\+", "zn2+", t)
    t = re.sub(r"pb\s*2\s*\+", "pb2+", t)

    # ADD-mode синонимы (не удаляем исходный токен)
    add_rules = [
        (r"\bh2so4\b", "серная кислота"),
        (r"\bcl-\b", "хлорид"),
        (r"\bcl2\b", "хлор газ"),
        (r"\bso4\b", "сульфат"),
        (r"\bzn2\+\b", "ионы цинка"),
        (r"\bpb2\+\b", "ионы свинца"),
        (r"\bni2\+\b", "ионы никеля"),
        (r"\bfe2\+\b", "ионы железа(II)"),
        (r"\bcu2\+\b", "ионы меди(II)"),
        (r"\bco2\+\b", "ионы кобальта(II)"),
    ]

    for pat, syn in add_rules:
        # добавляем синоним только если формула встретилась и синонима ещё нет
        if re.search(pat, t) and syn not in t:
            t = re.sub(pat, lambda m: m.group(0) + " " + syn, t)

    t = re.sub(r"\s+", " ", t).strip()
    return t

# -------------------------
# 2) E5 префиксы
# -------------------------
def wrap_e5(model_name: str, s: str, kind="query") -> str:
    return f"{kind}: {s}" if "multilingual-e5" in model_name else s

# -------------------------
# 3) БАЗОВЫЙ SEARCH (cosine через нормализованные эмбеддинги)
# -------------------------
def embed_norm(model, texts):
    emb = model.encode(texts, show_progress_bar=False).astype(np.float32)
    emb /= np.linalg.norm(emb, axis=1, keepdims=True) + 1e-12
    return emb

def search_with_emb(q_emb, doc_chunks, doc_emb, top_k=5):
    scores = (doc_emb @ q_emb.reshape(-1, 1)).ravel()
    idx = np.argsort(-scores)[:top_k]
    return [(int(i), float(scores[i]), doc_chunks[i]) for i in idx]

# -------------------------
# 4) РЕЖИМЫ RETRIEVAL
# -------------------------
bge_name = "BAAI/bge-m3"
e5_name = "intfloat/multilingual-e5-large"

bge = SentenceTransformer(bge_name)
e5  = SentenceTransformer(e5_name)

# embeddings документов (один раз)
doc_emb_bge = embed_norm(bge, doc_chunks)
doc_emb_e5  = embed_norm(e5,  [wrap_e5(e5_name, c, kind="passage") for c in doc_chunks])

def retrieve_bge(q, top_k=5):
    q_emb = embed_norm(bge, [q])[0]
    return search_with_emb(q_emb, doc_chunks, doc_emb_bge, top_k)

def retrieve_e5(q, top_k=5):
    q2 = wrap_e5(e5_name, q, kind="query")
    q_emb = embed_norm(e5, [q2])[0]
    return search_with_emb(q_emb, doc_chunks, doc_emb_e5, top_k)

def retrieve_bge_norm(q, top_k=5):
    q2 = canon_add_synonyms(q)
    q_emb = embed_norm(bge, [q2])[0]
    return search_with_emb(q_emb, doc_chunks, doc_emb_bge, top_k)

def retrieve_mix(q, top_k=5, w_bge=0.7, w_e5=0.3):
    qn = canon_add_synonyms(q)

    q_emb_b = embed_norm(bge, [qn])[0]
    s_b = (doc_emb_bge @ q_emb_b.reshape(-1,1)).ravel()

    q_emb_e = embed_norm(e5, [wrap_e5(e5_name, qn, kind="query")])[0]
    s_e = (doc_emb_e5 @ q_emb_e.reshape(-1,1)).ravel()

    scores = w_bge * s_b + w_e5 * s_e
    idx = np.argsort(-scores)[:top_k]
    return [(int(i), float(scores[i]), doc_chunks[i]) for i in idx]

# -------------------------
# 5) GOLD для chem_questions (семантический, по "ожидаемому ответу")
#    Вручную проще: почти все эти вопросы должны попасть в chunk "1.1 Сырьё" (id=1),
#    потому что там состав. Но сделаем автоматически через ответ-строку.
# -------------------------
chem_answers = [
    "Cl− 240–260",
    "SO4^2− 55–75",
    "Zn2+ не более 0.0025",
    "Pb2+ не более 0.032",
    "H2SO4 0.2–1.0",
]

def build_semantic_gold_by_answer(answers, model, doc_emb, model_name=None):
    # ответы кодируем как query (для e5 — с префиксом)
    texts = []
    for a in answers:
        a2 = wrap_e5(model_name, a, kind="query") if model_name else a
        texts.append(a2)
    a_emb = embed_norm(model, texts)
    gold = {}
    for i in range(len(answers)):
        scores = (doc_emb @ a_emb[i].reshape(-1,1)).ravel()
        gold[i+1] = int(np.argmax(scores))
    return gold

# gold для хим вопросов строим на bge (можно и на e5 — не критично)
gold_chem = build_semantic_gold_by_answer(chem_answers, bge, doc_emb_bge)

# -------------------------
# 6) ОЦЕНКА
# -------------------------
def eval_mode(questions, gold, retrieve_fn, name, ks=(1,3,5), top_k=5):
    hit = {k: 0 for k in ks}
    mrr = 0.0
    rows = []
    for qid, q in enumerate(questions, start=1):
        g = gold[qid]
        res = retrieve_fn(q, top_k=top_k)
        ids = [cid for cid,_,_ in res]
        for k in ks:
            hit[k] += int(g in ids[:k])
        if g in ids:
            mrr += 1.0 / (ids.index(g)+1)

        for rank,(cid,score,txt) in enumerate(res, start=1):
            rows.append({
                "mode": name, "qid": qid, "rank": rank, "score": score,
                "chunk_id": cid, "is_gold": (cid==g),
                "q": q, "preview": re.sub(r"\s+"," ",txt)[:160]
            })
    n = len(questions)
    metrics = {f"Recall@{k}": hit[k]/n for k in ks}
    metrics["MRR@5"] = mrr/n
    return metrics, pd.DataFrame(rows)

modes = [
    ("bge", retrieve_bge),
    ("e5", retrieve_e5),
    ("bge_norm", retrieve_bge_norm),
    ("mix", retrieve_mix),
]

# --- A) химический набор
metrics_rows = []
dfs = []
for name, fn in modes:
    metrics, df_dbg = eval_mode(chem_questions, gold_chem, fn, name, ks=(1,3,5), top_k=5)
    metrics_rows.append({"set":"chem", "mode": name, **metrics})
    dfs.append(df_dbg)

df_metrics = pd.DataFrame(metrics_rows).sort_values(["set","Recall@1"], ascending=[True, False])
display(df_metrics)

df_debug = pd.concat(dfs).sort_values(["mode","qid","rank"])
display(df_debug)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
